# 01 - EDA：圖書館借閱資料探索

目標：理解資料的分布，為後續推薦系統設計提供依據

In [ ]:
import sys, os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# 中文字型
plt.rcParams['font.sans-serif'] = ['Microsoft JhengHei', 'SimHei']
plt.rcParams['axes.unicode_minus'] = False
sns.set_theme(style='whitegrid', font='Microsoft JhengHei')

PROC = Path('../data/processed')
FIG  = Path('../results/figures')
FIG.mkdir(parents=True, exist_ok=True)

In [ ]:
borrows = pd.read_parquet(PROC / 'borrows.parquet')
users = pd.read_parquet(PROC / 'users.parquet')
books = pd.read_parquet(PROC / 'books.parquet')
reservations = pd.read_parquet(PROC / 'reservations.parquet')

print(f'借閱：{len(borrows):,}')
print(f'預約：{len(reservations):,}')
print(f'讀者：{len(users):,}')
print(f'書籍：{len(books):,}')
borrows.head()

## 1. 基本統計

In [ ]:
print('=== 缺值情形 ===')
print(borrows.isnull().sum())
print('\n=== 性別分布（借閱者）===')
print(borrows['gender'].value_counts(dropna=False))
print('\n=== 年齡分布（借閱者）===')
print(borrows['age'].describe())

## 2. 借閱數分布（每位讀者的借閱次數）

In [ ]:
user_cnt = borrows.groupby('user_id').size()
book_cnt = borrows.groupby('book_id').size()

print('每位讀者借閱次數：')
print(user_cnt.describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]))
print('\n每本書被借次數：')
print(book_cnt.describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].hist(np.clip(user_cnt.values, 0, 100), bins=50)
axes[0].set_title('每位讀者的借閱次數（截到100）')
axes[0].set_xlabel('借閱次數')
axes[0].set_ylabel('讀者人數')
axes[1].hist(np.clip(book_cnt.values, 0, 100), bins=50)
axes[1].set_title('每本書被借的次數（截到100）')
axes[1].set_xlabel('被借次數')
axes[1].set_ylabel('書籍數')
plt.tight_layout()
plt.savefig(FIG / 'fig01_interaction_dist.png', dpi=150)
plt.show()

## 3. 時間趨勢

In [ ]:
borrows['ts'] = pd.to_datetime(borrows['ts'])
borrows['date'] = borrows['ts'].dt.date
borrows['month'] = borrows['ts'].dt.to_period('M').astype(str)
borrows['hour'] = borrows['ts'].dt.hour
borrows['weekday'] = borrows['ts'].dt.day_name()

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
borrows.groupby('month').size().plot(kind='bar', ax=axes[0,0], color='steelblue')
axes[0,0].set_title('每月借閱量')
axes[0,0].set_xlabel('月份')

borrows.groupby('hour').size().plot(kind='bar', ax=axes[0,1], color='coral')
axes[0,1].set_title('小時別借閱量')
axes[0,1].set_xlabel('小時')

weekday_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
borrows.groupby('weekday').size().reindex(weekday_order).plot(kind='bar', ax=axes[1,0], color='seagreen')
axes[1,0].set_title('星期別借閱量')

borrows.groupby('date').size().plot(ax=axes[1,1], color='purple')
axes[1,1].set_title('每日借閱量趨勢')
axes[1,1].set_xlabel('日期')
plt.tight_layout()
plt.savefig(FIG / 'fig02_time_pattern.png', dpi=150)
plt.show()

## 4. 性別與年齡 × 借閱

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
borrows['gender'].value_counts().plot(kind='bar', ax=axes[0], color=['steelblue','pink','grey'])
axes[0].set_title('借閱者性別分布')

axes[1].hist(borrows['age'].dropna(), bins=50, color='coral')
axes[1].set_title('借閱者年齡分布')
axes[1].set_xlabel('年齡')
plt.tight_layout()
plt.savefig(FIG / 'fig03_demo.png', dpi=150)
plt.show()

## 5. 熱門類別（中圖法分類號取首位）

In [ ]:
category_label = {
    '0':'總類','1':'哲學類','2':'宗教類','3':'科學類','4':'應用科學',
    '5':'社會科學','6':'史地中國','7':'史地世界','8':'語文文學','9':'藝術類'
}
def cat_top(c):
    if pd.isna(c): return None
    s = str(c).strip()
    return s[0] if s and s[0].isdigit() else None

borrows['cat_top'] = borrows['category'].apply(cat_top)
ct = borrows['cat_top'].value_counts().sort_index()
ct.index = [f'{k} {category_label.get(k, "?")}' for k in ct.index]
ct.plot(kind='bar', figsize=(12,4), color='teal')
plt.title('各大類借閱量（中國圖書分類法）')
plt.tight_layout()
plt.savefig(FIG / 'fig04_category.png', dpi=150)
plt.show()

## 6. 互動稀疏度（推薦系統最重要指標）

In [ ]:
n_u = borrows['user_id'].nunique()
n_i = borrows['book_id'].nunique()
n_int = len(borrows.drop_duplicates(['user_id','book_id']))
density = n_int / (n_u * n_i)
print(f'讀者數：{n_u:,}')
print(f'書籍數：{n_i:,}')
print(f'unique 互動：{n_int:,}')
print(f'矩陣密度：{density*100:.5f}%（典型推薦系統 < 1%，這就是稀疏問題）')